In [1]:
import pandas as pd
import numpy as np
import os

print("="*80)
print("CREATING MASTER TABLE FOR PROJECT")
print("="*80)

# Path to processed data
processed_path = '../data/processed/'


print("\n Loading cleaned datasets...")

oil_price = pd.read_csv(f'{processed_path}Crud_Oil_Price_cleaned.csv')
cpih_inflation = pd.read_csv(f'{processed_path}uk_cpih_inflation_by_category_cleaned.csv')
fuel_energy = pd.read_csv(f'{processed_path}uk_cpih_fuel_energy_inflation_cleaned.csv')
cpih_yoy = pd.read_csv(f'{processed_path}uk_cpih_inflation_with_yoy_cleaned.csv')
petrol_diesel = pd.read_csv(f'{processed_path}uk_petrol_diesel_cleaned.csv')
household_income = pd.read_csv(f'{processed_path}uk_household_spending_by_income_group_cleaned.csv')
household_decile = pd.read_csv(f'{processed_path}uk_household_expenditure_by_decile_cleaned.csv')
fuel_transport = pd.read_csv(f'{processed_path}uk_fuel_transport_spending_by_decile_cleaned.csv')
petrol_diesel_spend = pd.read_csv(f'{processed_path}uk_petrol_diesel_spending_by_decile_cleaned.csv')

print(f"   ✅ Loaded 9 datasets")


print("\n Preparing oil price data...")

oil_price['Date'] = pd.to_datetime(oil_price['Date'])
oil_price['year_month'] = oil_price['Date'].dt.to_period('M')

# Aggregate to monthly
oil_monthly = oil_price.groupby('year_month').agg({
    'Price': 'mean',
    'Change %': 'mean',
    'High': 'mean',
    'Low': 'mean',
    'Vol.': 'mean'
}).reset_index()

# Rename columns
oil_monthly.columns = ['year_month', 'crude_oil_price', 'oil_price_change_pct', 
                       'oil_high', 'oil_low', 'oil_volume']

# Calculate volatility (using daily data within month)
oil_volatility = oil_price.groupby('year_month')['Price'].std().reset_index()
oil_volatility.columns = ['year_month', 'oil_price_volatility']

oil_monthly = oil_monthly.merge(oil_volatility, on='year_month', how='left')

# Calculate rolling averages and lags
oil_monthly = oil_monthly.sort_values('year_month')
oil_monthly['oil_price_lag_1'] = oil_monthly['crude_oil_price'].shift(1)
oil_monthly['oil_price_lag_3'] = oil_monthly['crude_oil_price'].shift(3)
oil_monthly['oil_price_lag_6'] = oil_monthly['crude_oil_price'].shift(6)
oil_monthly['rolling_avg_3m'] = oil_monthly['crude_oil_price'].rolling(3).mean()
oil_monthly['rolling_avg_6m'] = oil_monthly['crude_oil_price'].rolling(6).mean()
oil_monthly['oil_price_shock'] = oil_monthly['crude_oil_price'] - oil_monthly['rolling_avg_3m']

print(f"   ✅ Oil data prepared: {len(oil_monthly)} monthly records")


print("\n Preparing CPIH inflation data...")

cpih_inflation['Date'] = pd.to_datetime(cpih_inflation['Date'])
cpih_inflation['year_month'] = cpih_inflation['Date'].dt.to_period('M')

# Select key CPIH columns
cpih_subset = cpih_inflation[['year_month', 'overall_index_cpih', 'food_non_alcoholic_beverages_cpih',
                              'electricity_gas_fuels_cpih', 'transport_cpih', 
                              'housing_utilities_cpih', 'electricity_cpih', 'gas_cpih',
                              'liquid_fuels_cpih', 'solid_fuels_cpih']]

# Get YoY data
cpih_yoy['Date'] = pd.to_datetime(cpih_yoy['Date'])
cpih_yoy['year_month'] = cpih_yoy['Date'].dt.to_period('M')
cpih_yoy_subset = cpih_yoy[['year_month', 'overall_index_yoy', 'transport_fuels_lubricants_cpih_yoy']]

# Merge CPIH data
cpih_data = cpih_subset.merge(cpih_yoy_subset, on='year_month', how='left')

print(f"   ✅ CPIH data prepared: {len(cpih_data)} records")


print("\n Preparing fuel energy data...")

fuel_energy['date'] = pd.to_datetime(fuel_energy['date'])
fuel_energy['year_month'] = fuel_energy['date'].dt.to_period('M')

fuel_energy_subset = fuel_energy[['year_month', 'electricity_cpih', 'gas_cpih', 
                                  'liquid_fuels_cpih', 'solid_fuels_cpih',
                                  'transport_fuels_lubricants_cpih']]

print(f"   ✅ Fuel energy data prepared: {len(fuel_energy_subset)} records")


print("\n Preparing petrol/diesel price data...")

petrol_diesel['date'] = pd.to_datetime(petrol_diesel['date'])
petrol_diesel['year_month'] = petrol_diesel['date'].dt.to_period('M')

petrol_monthly = petrol_diesel.groupby('year_month').agg({
    'petrol_price_pence': 'mean',
    'diesel_price_pence': 'mean',
    'Price_Difference': 'mean'
}).reset_index()

print(f"   ✅ Petrol/diesel data prepared: {len(petrol_monthly)} records")


print("\n Preparing household spending data...")

#  Household spending by income group
income_spending = household_income.copy()
income_spending_pivot = income_spending.pivot_table(
    index=['year'],
    columns=['income_group', 'spending_category'],
    values='spending_percent'
).reset_index()
# Flatten column names
income_spending_pivot.columns = ['year'] + [f"{col[0]}_{col[1]}" for col in income_spending_pivot.columns[1:]]

#  Fuel transport spending by decile
fuel_transport_pivot = fuel_transport.pivot_table(
    index=['year'],
    columns=['decile', 'fuel_transport_category'],
    values='spending_percent'
).reset_index()
fuel_transport_pivot.columns = ['year'] + [f"{col[0]}_{col[1]}" for col in fuel_transport_pivot.columns[1:]]

#  Petrol diesel spending by decile
petrol_diesel_spend_pivot = petrol_diesel_spend.pivot_table(
    index=['year'],
    columns=['decile', 'fuel_type'],
    values='spending_percent'
).reset_index()
petrol_diesel_spend_pivot.columns = ['year'] + [f"{col[0]}_{col[1]}" for col in petrol_diesel_spend_pivot.columns[1:]]

print(f"   ✅ Household spending data prepared")


print("\n Creating master table...")

# Start with oil data
master = oil_monthly.copy()

# Add year and month
master['year'] = master['year_month'].dt.year
master['month'] = master['year_month'].dt.month

# Merge CPIH data (using consistent column name 'year_month')
master = master.merge(cpih_data, on='year_month', how='left')

# Merge fuel energy data
master = master.merge(fuel_energy_subset, on='year_month', how='left')

# Merge petrol/diesel data
master = master.merge(petrol_monthly, on='year_month', how='left')

# Merge household spending data (by year)
master = master.merge(income_spending_pivot, on='year', how='left')
master = master.merge(fuel_transport_pivot, on='year', how='left')
master = master.merge(petrol_diesel_spend_pivot, on='year', how='left')

print(f"   ✅ Master table created: {master.shape[0]} rows × {master.shape[1]} columns")


print("\n Feature engineering...")

#  Inflation adjusted oil price
master['inflation_adjusted_oil'] = master['crude_oil_price'] / master['overall_index_cpih']

#  Fuel inflation rate
master['fuel_inflation_rate'] = master['transport_fuels_lubricants_cpih_yoy']

#  Price change indicators
master['oil_price_change_pct'] = master['crude_oil_price'].pct_change() * 100
master['oil_price_change_3m'] = master['crude_oil_price'].pct_change(periods=3) * 100

#  Flags
oil_median = master['crude_oil_price'].median()
oil_75th = master['crude_oil_price'].quantile(0.75)

master['is_high_oil_price'] = (master['crude_oil_price'] > oil_median).astype(int)
master['is_energy_crisis'] = (master['crude_oil_price'] > oil_75th).astype(int)

#  Create income group specific features
for income_group in ['Lowest_20%', 'Middle_40%', 'Highest_20%']:
    # Find columns for this income group
    cols = [col for col in master.columns if income_group in col]
    if len(cols) > 0:
        master[f'total_spend_{income_group}'] = master[cols].sum(axis=1)
        # Calculate fuel spend ratio (using first fuel-related column found)
        fuel_cols = [col for col in cols if 'fuel' in col.lower() or 'electricity' in col.lower() or 'gas' in col.lower()]
        if len(fuel_cols) > 0:
            master[f'fuel_spend_ratio_{income_group}'] = master[fuel_cols[0]] / master[f'total_spend_{income_group}']

#  Create decile specific features
for decile in ['Lowest_10%', 'Second_10%', 'Third_10%', 'Fourth_10%', 'Fifth_10%',
               'Sixth_10%', 'Seventh_10%', 'Eighth_10%', 'Ninth_10%', 'Highest_10%']:
    # Find columns for this decile
    cols = [col for col in master.columns if decile in col]
    if len(cols) > 0:
        master[f'total_spend_{decile}'] = master[cols].sum(axis=1)

print(f"   ✅ Feature engineering complete: {master.shape[1]} columns")


print("\n Cleaning master table...")

# Remove columns with > 50% missing values
threshold = 0.5 * len(master)
master = master.dropna(thresh=threshold, axis=1)

# Fill remaining missing values
for col in master.select_dtypes(include=[np.number]).columns:
    if master[col].isnull().sum() > 0:
        master[col] = master[col].fillna(master[col].median())

# Drop duplicate columns
master = master.loc[:, ~master.columns.duplicated()]

print(f"   ✅ Master table cleaned: {master.shape[0]} rows × {master.shape[1]} columns")


print("\n Saving master table...")

master.to_csv(f'{processed_path}master_household_oil_impact.csv', index=False)
print(f"   ✅ Master table saved to: {processed_path}master_household_oil_impact.csv")

# ============================================
# STEP 11: Summary Report
# ============================================
print("\n" + "="*80)
print("MASTER TABLE CREATION COMPLETE")
print("="*80)

print(f"""
Master Table Summary:
---------------------
- Total Rows: {master.shape[0]:,}
- Total Columns: {master.shape[1]:,}
- Date Range: {master['year_month'].min()} to {master['year_month'].max()}
- Years Covered: {master['year'].min()} to {master['year'].max()}

Key Columns Included:
---------------------
- Oil Prices: crude_oil_price, oil_price_change_pct, oil_price_volatility
- Fuel Prices: petrol_price_pence, diesel_price_pence
- Inflation: overall_index_cpih, overall_index_yoy
- Spending: Various household spending columns by income group and decile
- Derived Features: inflation_adjusted_oil
- Flags: is_high_oil_price, is_energy_crisis

Sample Columns:
---------------""")

# Show sample columns
sample_cols = ['year_month', 'year', 'month', 'crude_oil_price', 'petrol_price_pence', 
               'diesel_price_pence', 'overall_index_cpih', 'overall_index_yoy',
               'inflation_adjusted_oil', 'is_high_oil_price']
available_cols = [col for col in sample_cols if col in master.columns]
print(master[available_cols].head(10))

print("\n" + "="*80)
print("✅ MASTER TABLE READY FOR EDA, FEATURE ENGINEERING, AND MODELING!")

CREATING MASTER TABLE FOR PROJECT

 Loading cleaned datasets...
   ✅ Loaded 9 datasets

 Preparing oil price data...
   ✅ Oil data prepared: 69 monthly records

 Preparing CPIH inflation data...
   ✅ CPIH data prepared: 457 records

 Preparing fuel energy data...
   ✅ Fuel energy data prepared: 457 records

 Preparing petrol/diesel price data...
   ✅ Petrol/diesel data prepared: 69 records

 Preparing household spending data...
   ✅ Household spending data prepared

 Creating master table...
   ✅ Master table created: 69 rows × 374 columns

 Feature engineering...
   ✅ Feature engineering complete: 393 columns

 Cleaning master table...
   ✅ Master table cleaned: 69 rows × 393 columns

 Saving master table...
   ✅ Master table saved to: ../data/processed/master_household_oil_impact.csv

MASTER TABLE CREATION COMPLETE

Master Table Summary:
---------------------
- Total Rows: 69
- Total Columns: 393
- Date Range: 2020-12 to 2026-08
- Years Covered: 2020 to 2026

Key Columns Included:
--